# EGNN-SBI Training on GPU (Pipeline C)

**What this notebook does:**
1. Installs dependencies (`sbi`)
2. Clones your repo from GitHub
3. Copies data from Google Drive
4. Trains EGNN + SNPE on GPU
5. Runs evaluation
6. Saves results back to Google Drive

**Before running:** Make sure you've:
- Pushed your latest code to GitHub (`cbharathulwar/sbi-srim`)
- Uploaded `mcpe_3d_train.csv` and `mcpe_3d_eval.csv` to a Google Drive folder called `sbi-srim-data/`

**Runtime settings:** Go to `Runtime > Change runtime type` and select **T4 GPU**. That's all you need.

## 1. Check GPU & Install Dependencies

In [ ]:
# Verify GPU is available
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
    print('ERROR: Not connected to a GPU!')
    print('Go to Runtime > Change runtime type > T4 GPU')
else:
    print(gpu_info)
    print('\n GPU is ready!')

In [ ]:
# Install sbi (pulls in torch, scipy, numpy, pandas automatically)
!pip install sbi tensorboard -q
print('Dependencies installed!')

## 2. Mount Google Drive & Clone Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted!')

In [ ]:
import os

# Clone repo (or pull latest if already cloned)
REPO_DIR = '/content/sbi-srim'
if os.path.exists(REPO_DIR):
    print('Repo already exists, pulling latest...')
    !cd {REPO_DIR} && git pull
else:
    !git clone -b feature/egnn-pipeline https://github.com/cbharathulwar/sbi-srim.git {REPO_DIR}
    print('Repo cloned!')

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')
!git branch

In [ ]:
# Copy data from Google Drive to local storage (MUCH faster I/O than reading from Drive)
# Adjust DRIVE_DATA_DIR if your CSVs are in a different folder
DRIVE_DATA_DIR = '/content/drive/MyDrive/sbi-srim-data'
LOCAL_DATA_DIR = f'{REPO_DIR}/data/mcpe3d'

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

import shutil
for csv_name in ['mcpe_3d_train.csv', 'mcpe_3d_eval.csv']:
    src = os.path.join(DRIVE_DATA_DIR, csv_name)
    dst = os.path.join(LOCAL_DATA_DIR, csv_name)
    if os.path.exists(dst):
        print(f'  {csv_name} already exists locally, skipping copy')
    elif os.path.exists(src):
        print(f'  Copying {csv_name} to local storage...')
        shutil.copy2(src, dst)
        print(f'  Done! ({os.path.getsize(dst) / 1e6:.1f} MB)')
    else:
        print(f'  ERROR: {src} not found!')
        print(f'  Upload {csv_name} to Google Drive folder: sbi-srim-data/')

print('\nData files:')
!ls -lh {LOCAL_DATA_DIR}/

## 3. Quick Sanity Check (Optional)
Run a quick test with 5k tracks to make sure everything works before committing to the full run.

In [ ]:
# Quick sanity check (~2-5 min on GPU)
# Skip this cell if you want to go straight to full training
!python -m src.scripts.egnn_3d --quick

## 4. Full Training Run
This is the real deal. On a T4 GPU, expect:
- Preprocessing: ~2-3 min (kNN precomputation)
- Training: ~1-2 hours (depends on convergence)
- Evaluation: ~5-10 min

In [ ]:
# Full training run on GPU
!python -m src.scripts.egnn_3d

## 5. Save Results to Google Drive
Copy the trained posterior and all results back to Drive so they persist after the session ends.

In [ ]:
# Save everything back to Google Drive
DRIVE_RESULTS_DIR = '/content/drive/MyDrive/sbi-srim-results'
LOCAL_RESULTS_DIR = f'{REPO_DIR}/results/egnn_3d'

if os.path.exists(LOCAL_RESULTS_DIR):
    # Copy entire results directory to Drive
    dst = DRIVE_RESULTS_DIR
    if os.path.exists(dst):
        shutil.rmtree(dst)  # Remove old results
    shutil.copytree(LOCAL_RESULTS_DIR, dst)
    print(f'Results saved to Google Drive: {dst}')
    print('\nSaved files:')
    for root, dirs, files in os.walk(dst):
        for f in files:
            fpath = os.path.join(root, f)
            size_mb = os.path.getsize(fpath) / 1e6
            print(f'  {os.path.relpath(fpath, dst)} ({size_mb:.1f} MB)')
else:
    print('ERROR: No results directory found. Did training complete?')

## 6. View Results

In [ ]:
import pandas as pd
from IPython.display import display

# Show eval results summary
eval_csv = f'{LOCAL_RESULTS_DIR}/egnn_3d_eval_results.csv'
if os.path.exists(eval_csv):
    df = pd.read_csv(eval_csv)
    print(f'Evaluated {len(df)} tracks')
    print(f'\nMedian Energy Error:   {df["energy_error"].median():.2f} keV')
    print(f'Median Angular Error:  {df["angular_error_deg"].median():.2f} deg')
    print(f'Head-Tail Flip Rate:   {(df["angular_error_deg"] > 90).mean() * 100:.2f}%')
    if 'energy_std' in df.columns:
        print(f'Median Energy sigma:   {df["energy_std"].median():.2f} keV')
    if 'angular_cone_68' in df.columns:
        print(f'Median 68% Cone:       {df["angular_cone_68"].median():.2f} deg')
else:
    print('No eval results found yet.')

In [ ]:
# Display saved plots
from IPython.display import Image, display
import glob

plot_files = sorted(glob.glob(f'{LOCAL_RESULTS_DIR}/*.png'))
for pf in plot_files:
    print(f'\n--- {os.path.basename(pf)} ---')
    display(Image(filename=pf, width=800))

In [ ]:
# Show training curve
train_log = f'{LOCAL_RESULTS_DIR}/training_log.csv'
if os.path.exists(train_log):
    import matplotlib.pyplot as plt
    df_log = pd.read_csv(train_log)
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    ax.plot(df_log['epoch'], df_log['train_loss'], label='Train Loss')
    ax.plot(df_log['epoch'], df_log['val_loss'], label='Val Loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('EGNN-SBI Training Curve')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('No training log found.')